# 01 — Exploración de Datos

Este notebook permite explorar los datasets descargados:
- Cargar el dataset y revisar estructura
- Ver distribución de clases
- Visualizar imágenes de ejemplo con sus anotaciones
- Verificar integridad de los datos

In [ ]:
# === Setup para Google Colab ===
import os

if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/neurodrive
    !pip install -q ultralytics albumentations
    print('✅ Entorno Colab configurado')
else:
    # En local, asegurar que estamos en el directorio del proyecto
    os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
    print('✅ Entorno local detectado')

In [ ]:
# === Configuración ===
# Cambiar estos valores según el dataset a explorar

DATASET_NAME = 'tier1_road_vehicles'  # nombre del dataset
DATASET_CONFIG = f'configs/datasets/{DATASET_NAME}.yaml'  # config YAML
DATA_DIR = 'data/raw'  # directorio de datos crudos

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import yaml
import cv2
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Cargar configuración del dataset
with open(DATASET_CONFIG, 'r', encoding='utf-8') as f:
    dataset_config = yaml.safe_load(f)

print(f"Dataset: {dataset_config['dataset']['name']}")
print(f"Formato: {dataset_config['dataset']['format']}")
print(f"Clases: {dataset_config['dataset']['classes']}")
print(f"Notas: {dataset_config['dataset']['notes']}")

## Estructura del Dataset

Revisamos la estructura de archivos del dataset descargado.

In [ ]:
dataset_path = Path(DATA_DIR) / DATASET_NAME

if not dataset_path.exists():
    print(f'⚠️ Dataset no encontrado en: {dataset_path}')
    print('Ejecuta primero: python scripts/download_datasets.py --dataset', DATASET_NAME)
else:
    # Listar archivos y directorios
    for item in sorted(dataset_path.rglob('*'))[:30]:
        relative = item.relative_to(dataset_path)
        prefix = '📁' if item.is_dir() else '📄'
        print(f'{prefix} {relative}')
    
    # Contar archivos por tipo
    extensions = Counter(f.suffix.lower() for f in dataset_path.rglob('*') if f.is_file())
    print(f'\nArchivos por tipo: {dict(extensions)}')

## Distribución de Clases

Analizamos cuántas instancias hay de cada clase en el dataset.

In [ ]:
# Buscar archivos de labels
labels_dir = None
for candidate in ['labels', 'train/labels', 'labels/train']:
    p = dataset_path / candidate
    if p.exists():
        labels_dir = p
        break

if labels_dir is None:
    print('⚠️ No se encontró directorio de labels')
else:
    # Contar clases
    class_counts = Counter()
    total_boxes = 0
    
    for label_file in labels_dir.glob('*.txt'):
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_counts[int(parts[0])] += 1
                    total_boxes += 1
    
    print(f'Total de labels: {len(list(labels_dir.glob("*.txt")))}')
    print(f'Total de bounding boxes: {total_boxes}')
    print(f'\nDistribución de clases:')
    for cls_id, count in sorted(class_counts.items()):
        print(f'  Clase {cls_id}: {count} ({count/total_boxes*100:.1f}%)')

In [ ]:
# Gráfica de distribución
if labels_dir and class_counts:
    fig, ax = plt.subplots(figsize=(10, 5))
    clases = [f'Clase {c}' for c in sorted(class_counts.keys())]
    conteos = [class_counts[c] for c in sorted(class_counts.keys())]
    
    bars = ax.bar(clases, conteos, color='#3498db')
    ax.set_title('Distribución de Clases en el Dataset')
    ax.set_ylabel('Cantidad de instancias')
    
    for bar, count in zip(bars, conteos):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

## Visualización de Muestras

Visualizamos algunas imágenes con sus bounding boxes para verificar la calidad de las anotaciones.

In [ ]:
# Buscar directorio de imágenes
images_dir = None
for candidate in ['images', 'train/images', 'images/train']:
    p = dataset_path / candidate
    if p.exists():
        images_dir = p
        break

if images_dir and labels_dir:
    # Tomar 6 imágenes de ejemplo
    sample_images = list(images_dir.glob('*.jpg'))[:6]
    if not sample_images:
        sample_images = list(images_dir.glob('*.png'))[:6]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Leer labels
        label_file = labels_dir / f'{img_path.stem}.txt'
        if label_file.exists():
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                        x1 = int((xc - bw/2) * w)
                        y1 = int((yc - bh/2) * h)
                        x2 = int((xc + bw/2) * w)
                        y2 = int((yc + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(img, f'Clase {cls_id}', (x1, y1-5),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name, fontsize=9)
        axes[idx].axis('off')
    
    # Ocultar ejes sobrantes
    for idx in range(len(sample_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Muestras del Dataset con Anotaciones', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No se encontraron directorios de imágenes o labels')

## Validación de Integridad

Verificamos que el dataset sea consistente: que cada imagen tenga su label y viceversa.

In [ ]:
from src.data.validate import validate_dataset

if images_dir and labels_dir:
    stats = validate_dataset(
        images_dir=str(images_dir),
        labels_dir=str(labels_dir),
        verbose=True,
    )
else:
    print('⚠️ Configura images_dir y labels_dir manualmente para validar')

## Siguiente Paso

Una vez explorado el dataset, continúa con:
- **02_entrenamiento_baseline.ipynb**: Entrenar el primer modelo YOLO